In [1]:
from jetbot import Camera, bgr8_to_jpeg
import jetson_inference
import jetson_utils
import ipywidgets.widgets as widgets
import traitlets
from IPython.display import display
import numpy as np
import cv2

# Load model
net = jetson_inference.detectNet("ssd-mobilenet-v2", threshold=0.1)

# Camera + display widget
camera = Camera.instance(width=300, height=300)
image_widget = widgets.Image(format='jpeg', width=300, height=300)
count_label = widgets.Label(value="Detections: 0")
display(widgets.VBox([image_widget, count_label]))

# Only show these COCO classes
TARGET_CLASSES = {
    44: "bottle",   # catches bottles AND most cans
    47: "cup",      # catches some cans/cups
}

def process_frame(change):
    frame = change['new']  # BGR numpy array from jetbot camera
    
    # Convert BGR → RGBA for jetson-inference
    rgba = np.zeros((frame.shape[0], frame.shape[1], 4), dtype=np.uint8)
    rgba[:, :, 0] = frame[:, :, 2]  # R
    rgba[:, :, 1] = frame[:, :, 1]  # G
    rgba[:, :, 2] = frame[:, :, 0]  # B
    rgba[:, :, 3] = 255
    
    # Upload to GPU
    cuda_img = jetson_utils.cudaFromNumpy(rgba)
    
    # Run detection WITHOUT auto-overlay (we'll draw only targets ourselves)
    detections = net.Detect(cuda_img, overlay="none")
    
    # Filter for bottles/cups only
    targets = [d for d in detections if d.ClassID in TARGET_CLASSES]
    
    # Draw boxes ourselves on the BGR frame
    output = frame.copy()
    for d in targets:
        x1, y1, x2, y2 = int(d.Left), int(d.Top), int(d.Right), int(d.Bottom)
        label = f"{TARGET_CLASSES[d.ClassID]} {d.Confidence:.2f}"
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output, label, (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    image_widget.value = bgr8_to_jpeg(output)
    count_label.value = f"Detections: {len(targets)} (bottles/cans/cups)"

camera.observe(process_frame, names='value')

RuntimeError: Could not initialize camera.  Please see error trace.